# Titanic Classification

Build a complete classification pipeline: problem definition, EDA, preprocessing, model training, prediction and evaluation.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("titanic")
df.head()

## 1. Problem understanding

Target: `survived` (0 = did not survive, 1 = survived). We will use passenger characteristics to predict survival.

## 2. EDA

In [ ]:
print(df.shape)
display(df.describe(include="all").T)

sns.countplot(data=df, x="survived")
plt.title("Target distribution")
plt.show()

sns.barplot(data=df, x="sex", y="survived")
plt.title("Survival rate by sex")
plt.show()

## 3. Preprocessing and train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df[features]
y = df["survived"]

num_cols = ["pclass", "age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 4. Logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

## 5. Evaluation – confusion matrix and classification report

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Titanic – Confusion Matrix")
plt.show()

### Conclusion
The notebook demonstrates the full supervised-learning pipeline. The exact scores are generated when the notebook is run and should be reported from the actual run rather than copied from an example.